# Load ENV Variable

In [6]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-6"

In [27]:
def add_user_message(messages, text):
    messages.append({
        "role": "user",
        "content": text
    })

def add_assistant_message(messages, text):
    messages.append({
        "role": "assistant",
        "content": text
    })

def chat(messages, system=None, stop_sequences=None, temperature=1.0):

    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    
    return message.content[0].text


In [31]:
messages = []

# add_assistant_message(messages, "```json")
add_user_message(messages, "Generate a very short event bridge rule as json")

# following not working as later models have dropped this support 
# add_assistant_message(messages, "```json")

system = "You are a JSON-only assistant. Every response must be raw, valid JSON with no markdown. Your response must start with { and end with }"

text = chat(messages, system=system, stop_sequences=["```"])
text

'{\n  "Name": "MyEventBridgeRule",\n  "EventPattern": {\n    "source": ["aws.ec2"],\n    "detail-type": ["EC2 Instance State-change Notification"]\n  },\n  "State": "ENABLED",\n  "Targets": [\n    {\n      "Id": "MyTarget",\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction"\n    }\n  ]\n}'

In [32]:
import json
json.loads(text.strip())

{'Name': 'MyEventBridgeRule',
 'EventPattern': {'source': ['aws.ec2'],
  'detail-type': ['EC2 Instance State-change Notification']},
 'State': 'ENABLED',
 'Targets': [{'Id': 'MyTarget',
   'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:MyFunction'}]}

# Exercise

In [ ]:
messages = []

prompt = """Generate three different sample AWS CLI commands. Each should be very short"""

add_user_message(messages, prompt)
add_assistant_message(messages, "Here are all three commands in a single block without any comments")
add_user_message(messages, "```bash")

# system = "You are a JSON-only assistant. Every response must be raw, valid JSON with no markdown. Provide all commands in a single response"

text = chat(messages) 
text.strip()



'```bash\naws s3 ls\naws ec2 describe-instances\naws iam list-users\n```'

In [65]:
from IPython.display import Markdown
Markdown(text)

```bash
aws s3 ls
aws ec2 describe-instances
aws iam list-users
```